In [1]:
# import python libraries
import sys
from opentrons import protocol_api, execute, simulate
from opentrons.types import Location, Point
import pandas as pd
import numpy as np
import urllib.request
import time
import threading
from datetime import datetime, timedelta

In [2]:
def labware_protocol(protocol):
    tiprack20 = protocol.load_labware('opentrons_96_tiprack_20ul', '8')
    pip20 = protocol.load_instrument('p20_single_gen2', 'left', tip_racks=[tiprack20])
    
    stock = protocol.load_labware('corning_24_wellplate_3.4ml_flat',location='4')
    
    # Add heater Shaker module and its plate
    hs_mod = protocol.load_module(module_name="heaterShakerModuleV1", location="3")
    hs_adapter = hs_mod.load_adapter("opentrons_universal_flat_adapter")
    hs_plate = hs_adapter.load_labware("corning_384_wellplate_112ul_flat")
    
    return tiprack20, pip20 ,stock, hs_mod, hs_plate


# Protocol function
def mixing(protocol,data_frame, pip300, plate, stock):
    '''
    stock: array of stock locations 
    well: array of well locations
    volume: array of volumes to aspirate and dispense
    '''
    # Get the data from the CSV file
    vol_1 = data_frame['Additive Volume'].tolist()
    vol_2 = data_frame['BAI Volume'].tolist() 
    vol_3 = data_frame['MAI Volume'].tolist() 
    vol_4 = data_frame['PbI2 Volume'].tolist() 
    
    wells = data_frame['well'].tolist()
    
    # Protocol 
    protocol.home()
    pip300.distribute(vol_1, stock.wells('A1'), [plate.well(well_id).top(-1) for well_id in wells])   # plate.columns()[0],plate.rows()[0]
    pip300.distribute(vol_2, stock.wells('B1'), [plate.well(well_id).top(-1) for well_id in wells]) 
    pip300.distribute(vol_3, stock.wells('C1'), [plate.well(well_id).top(-1) for well_id in wells]) 
    pip300.distribute(vol_4, stock.wells('D1'), [plate.well(well_id).top(-1) for well_id in wells]) 
    
    # Mix the wells with new tips
    for i in range(len(vol_1)):
        # Pick up a new tip
        pip300.pick_up_tip()
        # Mix the source well before transferring
        pip300.mix(10, 150, plate.well(wells[i]).bottom(0.2))  
        # Drop tip after 
        pip300.drop_tip()

    print("✅ Mixing complete!")
    for line in protocol.commands():
        print(line)

def start_heater(protocol, hs_mod):
    '''
    heater_mod: heater module from labware
    heater_plate: heater plate from labware
    '''
    # Close the opentron flap and start heater
    hs_mod.close_labware_latch()
    hs_mod.set_and_wait_for_temperature(95)
    print("✅ Heater Ready!")
    for line in protocol.commands():
        print(line)

def dropcasting(protocol,stock, pip20,hs_plate):
    '''
    mix_well: array of stock locations
    hs_well: array of well locations
    volume: integer volume, Recommneded 2ul
    '''
    # Get the data from the CSV file
    volume = 1.5

    #['C4', 'C10', 'C16', 'C21', 'H4', 'H10', 'H16', 'H21', 'N4', 'N10', 'N16','N21'] # of the 384 well plate
    hs_well = ['C4', 'H4'] # of the 384 well plate
    stock_well = ['A1','A1']
    
    protocol.home()
    for src,dest in zip(stock_well,hs_well):
        # Mix the source well before transferring
        # pip20.mix(10, 20, plate.wells_by_name()[src])  
        # Transfer once from mix_plate to hs_plate
        pip20.transfer(volume, stock.wells(src)[0].bottom(0.4), hs_plate.wells_by_name()[dest].bottom(0.2), new_tip='always', blow_out=True, blowout_location = "destination well")
        # Start timer or log sample
        # Drop the tip

    print("✅ Complete Dropcasting")
    for line in protocol.commands():
        print(line)

# Store all timers
active_timers = {}

class TimerThread(threading.Thread):
    def __init__(self, material_id, duration_min):
        super().__init__()
        self.material_id = material_id
        self.duration = timedelta(minutes=duration_min)
        self.start_time = datetime.now()
        self.end_time = self.start_time + self.duration

    def run(self):
        time.sleep(self.duration.total_seconds())
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ⏰ Material {self.material_id} is done!")

def add_material(material_id, duration_min):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ⏳ Timer started for {material_id} ({duration_min} min)")
    thread = TimerThread(material_id, duration_min)
    active_timers[material_id] = thread
    thread.start()


In [4]:
protocol2 = execute.get_protocol_api('2.19') # simulate.get_protocol_api('2.15') # 


/data/robot_settings.json not found. Loading defaults
Failed to initialize character device, will not be able to control gpios (lights, button, smoothiekill, smoothie reset). Only one connection can be made to the gpios at a time. If you need to control gpios, first stop the robot server with systemctl stop opentrons-robot-server. Until you restart the server with systemctl start opentrons-robot-server, you will be unable to control the robot using the Opentrons app.
/data/deck_calibration.json not found. Loading defaults
Exception in Thread Manager build
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/thread_manager.py", line 314, in _build_and_start_loop
  File "/usr/lib/python3.10/asyncio/base_events.py", line 649, in run_until_complete
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/api.py", line 248, in build_hardware_controller
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/module_c

ThreadManagerException: Failed to create Managed Object

In [ ]:
tiprack20, pip20 , stock, hs_mod, hs_plate = labware_protocol(protocol2)

#start_heater(protocol2, hs_mod)

NameError: name 'protocol2' is not defined

In [ ]:
dropcasting(protocol2,stock, pip20,hs_plate)

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 62, in parse_temperature_response
KeyError: 'T'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 400, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 409, in read_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 127, in get_temperature
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/utils.py", line 65, in parse_temperature_response
opentrons.drivers.utils.ParseError: ParseError(error_message=Unexpected argument to parse_temperature_response, parse_source=M241 STATUS:IDLE_CLOSED)

✅ Complete Dropcasting
Latching labware on Heater-Shaker
Setting Target Temperature of Heater-Shaker to 95 °C
Waiting for Heater-Shaker to reach target temperature
Transferring 1.5 from A1 of Corning 24 Well Plate 3.4 mL Flat on slot 4 to C4 of Corning 384 Well Plate 112 µL Flat on Opentrons Universal Flat Heater-Shaker Adapter on Heater-Shaker Module GEN1 on slot 3
Picking up tip from A1 of Opentrons OT-2 96 Tip Rack 20 µL on slot 8
Aspirating 1.5 uL from A1 of Corning 24 Well Plate 3.4 mL Flat on slot 4 at 7.56 uL/sec
Dispensing 1.5 uL into C4 of Corning 384 Well Plate 112 µL Flat on Opentrons Universal Flat Heater-Shaker Adapter on Heater-Shaker Module GEN1 on slot 3 at 7.56 uL/sec
Blowing out at C4 of Corning 384 Well Plate 112 µL Flat on Opentrons Universal Flat Heater-Shaker Adapter on Heater-Shaker Module GEN1 on slot 3
Dropping tip into Trash Bin on slot 12
Transferring 1.5 from A1 of Corning 24 Well Plate 3.4 mL Flat on slot 4 to C4 of Corning 384 Well Plate 112 µL Flat on Ope

Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 145, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 409, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 433, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 487, in _send_data
opentrons.drivers.asyncio.communication.errors.NoResponse: /dev/ot_module_heatershaker0: No response to 'M123 
'
Pol

### Notes
- In Drop casting, lower the pipet (not  up)
- In mixing, raise raise the pipet (cross contamination is happening...)
- Conduct mixing wells with 300 ul tip instead.
- Test protocol

In [6]:
# processed_ids = set()  # This should persist during a single script run

# while True:
#     try:
#         df = pd.read_csv(path)  # Read the CSV file again

#         for index, row in df.iterrows():
#             row_id = row["id"]
#             if row_id in processed_ids:
#                 continue  # Skip already-done rows

#             print("Made ", well, "with ", volume)
            
#             mixing(well, volume)
#             dropcasting(well, hs_well, 5)
            
#             processed_ids.add(row_id)

#             if "exit" in row.values:
#                 print("exit success")
#                 break

#         time.sleep(5*60)  # Wait for n*60 seconds before checking again (aka: n minutes)

#     except Exception as e:
#         print("Error reading file", e)
#         break


In [ ]:
protocol = simulate.get_protocol_api('2.15') # 
tiprack300 = protocol.load_labware('opentrons_96_tiprack_300ul', '7')
pip300 = protocol.load_instrument('p300_single_gen2', 'right', tip_racks=[tiprack300])

stock = protocol.load_labware('corning_24_wellplate_3.4ml_flat',location='4')
plate = protocol.load_labware('nest_96_wellplate_200ul_flat','5')


/data/robot_settings.json not found. Loading defaults
/data/deck_calibration.json not found. Loading defaults
Polling exception
Traceback (most recent call last):
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/poller.py", line 98, in _poll_once
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 401, in read
  File "/usr/lib/python3.10/site-packages/opentrons/hardware_control/modules/heater_shaker.py", line 412, in read_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/heater_shaker/driver.py", line 145, in get_rpm
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 409, in send_command
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 433, in send_data
  File "/usr/lib/python3.10/site-packages/opentrons/drivers/asyncio/communication/serial_connection.py", line 487, in _send_data
opentr